# Outcome Labels and Partitions

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
BASE = Path.cwd()
if not (BASE / "evidence").exists():
    BASE = BASE / "notebooks" 
assert (BASE / "evidence").is_dir(), "Open from the project root or notebooks."
EVIDENCE = BASE / "evidence"
def read_json(name):
    return json.loads((EVIDENCE / name).read_text(encoding="utf-8-sig"))
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})


## Prediction question
**Will a pest-related corrective return be recorded within 30 days after this paid service?** Yes = 1; No = 0. A normal next paid visit does not become a return label. A recorded corrective visit is not independent proof of pest recurrence or a verified formal warranty claim. Cancellations and unresolved/immature outcomes must not become negative examples.

## Business rules and model labels
Commercial clients have no warranty entitlement. Residential 1x has none. A confirmed 1x plus Upsell 2x is treated as a linked 3-service programme. Residential 3x allows the confirmed post-third-service warranty window; a scheduling delay beyond 14 days does not automatically remove eligibility. Residential 4x, 6x and 12x cover the service period plus 30 days after the final visit, regardless of pest. These entitlement rules and the model's recorded-return target are related but different concepts.

In [2]:
metadata = read_json("review_metadata.json")
display(pd.DataFrame([
    ["Earlier development pool (EDA)", metadata["eda_services"], metadata["eda_returns"]],
    ["Maturity gap (not added to earlier EDA or held-out assessment)", metadata["maturity_gap_services"], None],
    ["Historical assessment", metadata["holdout_services"], metadata["holdout_returns"]],
    ["Final later prospective refit (includes all mature input)", metadata["total_refit_services"], metadata["total_refit_returns"]]
], columns=["Role", "Services", "Recorded positive windows"]))
assert metadata["eda_services"] + metadata["maturity_gap_services"] + metadata["holdout_services"] == metadata["total_refit_services"]
display(pd.read_csv(EVIDENCE / "development_partitions.csv"))

Role,Services,Recorded positive windows
Earlier development pool (EDA),5333,492.0
Maturity gap (not added to earlier EDA or held-out assessment),262,NaN
Historical assessment,100,5.0
Final later prospective refit (includes all mature input),5695,521.0


period,validation_start,validation_end_exclusive,fit_services,validation_services,validation_returns,embargoed_training_rows,property_purged_training_rows
2025_q2,2025-04-01,2025-07-01,828,878,35,190,124
2025_q3,2025-07-01,2025-10-01,1567,877,78,258,195
2025_q4,2025-10-01,2026-01-01,2434,723,116,191,272


## Why the assessment has 100 services
The 100 are the reserved later-date assessment, not the entire dataset or 100 warranty visits. Five had recorded returns; 95 did not. The prospective refit was fitted later on 5,695 mature services; it must not be evaluated on those fitting rows as if they were unseen.

Development uses chronological folds with outcome embargo and property-group purging. The archived audit reports 34 development/assessment shared groups: the later historical assessment is not an entirely unseen-property test. Other larger historical studies are catalogued separately; they cannot be pooled as one untouched test.

## Future evaluation
The recovery cohort is 16–30 October, with earliest full outcome evaluation 30 November, conditional on complete timely predictions and outcomes. Missed or late predictions remain missing. This notebook does not check current collection health or claim a completed future result. Exact protocol: `config/cp2_recovery_cohort.json`.